# Phase 1–2: phenotype, data readiness, and cohort review

This notebook audits the available NIS variables and constructs the cached adult hematologic-malignancy hospitalization cohort. The phenotype is **draft pending clinical review**. No adjusted modeling is performed here.

Choose **Run → Run All Cells**. The first run builds the local cohort; unchanged reruns use the cache.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src/phase_1_2.py').is_file():
    raise RuntimeError('Open this notebook from the IPC-MPC-Study repository.')
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from src.phase_1_2 import main
result = main([])

## Phase 1: data-readiness audit

In [ ]:
output_dir = repo_root / 'outputs/phase_1_2'
audit = pd.read_csv(output_dir / 'data_readiness_audit.csv')
display(audit)
print('Copy/paste CSV:\n' + audit.to_csv(index=False))

## Decisions required before freezing the phenotype

In [ ]:
review = json.loads((output_dir / 'phenotype_review.json').read_text())
display(Markdown(f"**Version:** {review['version']}  \n**Status:** {review['review_status']}"))
display(Markdown('\n'.join(f"{i}. {item}" for i, item in enumerate(review['decisions_required_before_freeze'], 1))))

In [ ]:
phenotype = json.loads((repo_root / 'config/hm_phenotype_v0_1.json').read_text())
rules = pd.DataFrame([
    {
        'subtype': item['label'],
        'include_prefixes': ', '.join(item['include_prefixes']),
        'excluded_exact_codes': ', '.join(item.get('exclude_exact', [])) or 'None',
        'review_note': item.get('review_note', ''),
    }
    for item in phenotype['subtypes']
])
display(Markdown('### Draft HM subtype rules'))
display(rules)
print('Copy/paste CSV:\n' + rules.to_csv(index=False))

## Phase 2: adult HM cohort results

Counts refer to inpatient discharge records, not unique patients. Weighted totals covering all seven years are cumulative national discharge estimates, not annual counts.

In [ ]:
summary = json.loads((output_dir / 'cohort_summary.json').read_text())
summary_table = pd.DataFrame({'measure': list(summary), 'value': list(summary.values())})
display(summary_table)
print('Copy/paste CSV:\n' + summary_table.to_csv(index=False))

In [ ]:
by_year = pd.read_csv(output_dir / 'cohort_by_year.csv')
display(Markdown('### Cohort by year'))
display(by_year)
print('Copy/paste CSV:\n' + by_year.to_csv(index=False))

In [ ]:
by_subtype = pd.read_csv(output_dir / 'cohort_by_subtype.csv')
display(Markdown('### First-listed mutually exclusive HM subtype'))
display(by_subtype)
print('Copy/paste CSV:\n' + by_subtype.to_csv(index=False))

In [ ]:
overlap = pd.read_csv(output_dir / 'cohort_overlap.csv')
display(Markdown('### Number of distinct HM groups per hospitalization'))
display(overlap)
print('Copy/paste CSV:\n' + overlap.to_csv(index=False))

## Review checkpoint

Before proceeding, review whether cohort size, yearly stability, subtype distribution, overlap frequency, sepsis prevalence, and palliative-care prevalence are clinically plausible. Phase 3 or later modeling should wait until the phenotype decisions above are resolved.